<a href="https://colab.research.google.com/github/imagra93/ML-course-labs/blob/main/machina_learning/lab%20-%20Linear%20Regression%20for%20Time%20Series.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear Regression for Time Series

Until now we assumed the examples are **independent** (i.i.d.). A time series breaks that: the observations are ordered and each one is correlated with its own past.

The **model does not change**: it is still $\hat{y} = \boldsymbol{\theta}^\top\mathbf{x}$ fitted by least squares (notes 01). What changes is the **method**:

1. we can't split at random,
2. features may only use the past,
3. we compare against a naive forecast, not against $R^2$,
4. cross-validation must respect time.

Each of these mistakes gives a great validation score and a useless model. We build a daily-demand forecaster step by step and see each mistake happen.

**The model we will fit** predicts tomorrow from recent values (lags) and the calendar:

$$\hat{y}_{t} = \theta_0 + \underbrace{\sum_{i} \theta_i\, y_{t-i}}_{\text{lags}} + \underbrace{\beta\, t}_{\text{trend}} + \underbrace{\sum_{d} \gamma_d\, \mathbb{1}[\text{weekday} = d]}_{\text{weekly pattern}} + \underbrace{\sum_{j} \delta_j\, \mathbb{1}[\text{month} = j]}_{\text{annual pattern}}$$

Every feature must be **known before day $t$**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import TimeSeriesSplit, KFold, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## The Dataset

We simulate four years of daily demand, so we know the true process:

$$y_t = \underbrace{100 + 0.03\,t}_{\text{trend}} + \underbrace{s_{\text{weekday}(t)}}_{\text{weekly}} + \underbrace{15 \sin\!\left(\tfrac{2\pi t}{365.25} - 1.2\right)}_{\text{annual}} + \underbrace{\eta_t}_{\text{noise}}$$

The noise is **autocorrelated**, $\eta_t = 0.65\,\eta_{t-1} + \varepsilon_t$: today's noise carries over into tomorrow, so consecutive days look alike.

In [ ]:
N_DAYS = 4 * 365
dates = pd.date_range('2019-01-01', periods=N_DAYS, freq='D')
t = np.arange(N_DAYS)

# --- deterministic components -------------------------------------------------
trend = 100 + 0.03 * t

dow_effect = np.array([-4.0, -3.0, -1.0, 0.0, 6.0, 12.0, 8.0])   # Mon ... Sun
weekly = dow_effect[dates.dayofweek.to_numpy()]

yearly = 15 * np.sin(2 * np.pi * t / 365.25 - 1.2)

# --- autocorrelated noise: today's noise depends on yesterday's ---------------
PHI, SIGMA = 0.65, 5.0
eps = rng.normal(0, SIGMA, N_DAYS)
eta = np.zeros(N_DAYS)
for i in range(1, N_DAYS):
    eta[i] = PHI * eta[i - 1] + eps[i]

y = trend + weekly + yearly + eta

series = pd.DataFrame({'y': y, 't': t}, index=dates)
series.index.name = 'date'

print('--- Series ---')
print(f'  Period : {series.index.min().date()} to {series.index.max().date()}')
print(f'  Length : {len(series)} daily observations')
print(f'  Mean   : {series["y"].mean():.2f}    Std: {series["y"].std():.2f}')
series.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(series.index, series['y'], lw=0.9)
axes[0, 0].set_title('Daily demand — full series', fontsize=13)

# Zoom on eight weeks: the weekly pattern is only visible at this scale
zoom = series.iloc[200:256]
axes[0, 1].plot(zoom.index, zoom['y'], marker='o', ms=4, lw=1.2)
axes[0, 1].set_title('Zoom — eight weeks (the weekly cycle)', fontsize=13)
axes[0, 1].tick_params(axis='x', rotation=30)

by_dow = series.groupby(series.index.dayofweek)['y'].mean()
axes[1, 0].bar(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], by_dow.values,
               color='steelblue', edgecolor='white')
axes[1, 0].set_title('Mean demand by day of week', fontsize=13)

# Today against yesterday: the clearest possible picture of autocorrelation
axes[1, 1].scatter(series['y'].shift(1), series['y'], s=8, alpha=0.4)
axes[1, 1].set_xlabel('y yesterday', fontsize=11)
axes[1, 1].set_ylabel('y today', fontsize=11)
corr = series['y'].corr(series['y'].shift(1))
axes[1, 1].set_title(f'Today vs. yesterday (correlation {corr:.2f})', fontsize=13)

plt.tight_layout()
plt.show()

The last panel is the key to the whole lab: **if you know yesterday, you already know most of today.** That is great for forecasting (lag features use it), but it is also a trap, as we see next.

---
## Mistake 1: Splitting at Random

Same model, same features (`t`, yesterday, last week), two splits:

- **A:** `train_test_split(..., shuffle=True)`, as in the previous labs.
- **B:** the last 20% of the series, in time order.

In [ ]:
# A deliberately simple feature set for this demonstration: yesterday and last week.
demo = series.copy()
demo['lag_1'] = demo['y'].shift(1)
demo['lag_7'] = demo['y'].shift(7)
demo = demo.dropna()

X_demo = demo[['t', 'lag_1', 'lag_7']].to_numpy()
y_demo = demo['y'].to_numpy()

# --- Split A: random (INVALID for time series) --------------------------------
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
    X_demo, y_demo, test_size=0.2, shuffle=True, random_state=RANDOM_STATE
)
pred_a = LinearRegression().fit(Xa_tr, ya_tr).predict(Xa_te)

# --- Split B: chronological (VALID) -------------------------------------------
cut = int(len(X_demo) * 0.8)
Xb_tr, Xb_te = X_demo[:cut], X_demo[cut:]
yb_tr, yb_te = y_demo[:cut], y_demo[cut:]
pred_b = LinearRegression().fit(Xb_tr, yb_tr).predict(Xb_te)

print('--- Same model, same features, two different splits ---')
print(f'{"":<26}{"RMSE":>10}{"MAE":>10}{"R2":>10}')
for name, yt, yp in [('A  random split (WRONG)', ya_te, pred_a),
                     ('B  chronological split ', yb_te, pred_b)]:
    print(f'{name:<26}{mean_squared_error(yt, yp) ** 0.5:>10.3f}'
          f'{mean_absolute_error(yt, yp):>10.3f}{r2_score(yt, yp):>10.3f}')

The random split looks better, but only because it cheats: for a test day $t$, the training set contains days $t-1$ and $t+1$, which are almost the same value. It even trains on the future. The chronological score is the honest one.

> ### Rule 1: never shuffle a time series.

We use two years for training, one for validation (to choose the model) and one for test (used once, at the end).

In [ ]:
TRAIN_END = '2020-12-31'
VAL_END   = '2021-12-31'

train_idx = series.index <= TRAIN_END
val_idx   = (series.index > TRAIN_END) & (series.index <= VAL_END)
test_idx  = series.index > VAL_END

print('--- Chronological split ---')
for name, mask in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    sub = series.index[mask]
    print(f'  {name:<6}{len(sub):>5} days   {sub.min().date()} -> {sub.max().date()}')

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(series.index[train_idx], series['y'][train_idx], lw=0.9, label='train')
ax.plot(series.index[val_idx],   series['y'][val_idx],   lw=0.9, label='validation')
ax.plot(series.index[test_idx],  series['y'][test_idx],  lw=0.9, label='test')
ax.axvline(pd.Timestamp(TRAIN_END), color='black', ls='--', lw=1)
ax.axvline(pd.Timestamp(VAL_END),   color='black', ls='--', lw=1)
ax.set_title('Chronological train / validation / test split', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## Baselines and Metrics

Before fitting anything, measure what "doing nothing" scores:

| Baseline | Prediction |
|----------|------------|
| **Naive** | $\hat{y}_{t} = y_{t-1}$ (yesterday) |
| **Seasonal naive** | $\hat{y}_{t} = y_{t-7}$ (same day last week) |
| **Mean** | $\hat{y}_{t} = \bar{y}_{\text{train}}$ |

To compare we use **MASE** = MAE of the model / MAE of the seasonal naive forecast on the training set. **MASE < 1 beats the baseline, MASE > 1 loses to it.**

$R^2$ is misleading here: it compares against predicting the mean, which is a terrible forecast for a series with trend and seasonality, so almost anything gets a high $R^2$.

In [ ]:
y_all = series['y']
val_true = y_all[val_idx]

# Every baseline is causal: the prediction for day t uses only days before t.
baseline_preds = {
    'Naive (y[t-1])':          y_all.shift(1),
    'Seasonal naive (y[t-7])': y_all.shift(7),
    'Train mean':              pd.Series(y_all[train_idx].mean(), index=y_all.index),
}

# MASE denominator: seasonal-naive MAE on the TRAINING block only (a fixed number).
SEASON = 7
y_train = y_all[train_idx].to_numpy()
MASE_SCALE = np.mean(np.abs(y_train[SEASON:] - y_train[:-SEASON]))
print(f'MASE scale (seasonal-naive MAE on train): {MASE_SCALE:.4f}\n')


def evaluate(y_true, y_pred, name='model'):
    """Return the forecasting metrics as a dict."""
    err = np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float)
    return {
        'model': name,
        'MAE':  np.mean(np.abs(err)),
        'RMSE': np.sqrt(np.mean(err ** 2)),
        'MASE': np.mean(np.abs(err)) / MASE_SCALE,
        'R2':   r2_score(y_true, y_pred),
    }


pd.DataFrame([evaluate(val_true, pred[val_idx], name)
              for name, pred in baseline_preds.items()]).set_index('model').round(3)

Compare the `R2` and `MASE` columns: the seasonal naive has a respectable $R^2$ but a MASE of about 1. It is simply the bar to beat. (It is 1.04 and not exactly 1 because the denominator is measured on train and this row on validation.)

> ### Rule 2: compare against a real baseline, with MASE.

---
## Mistake 2: Features that Look at the Future

A 7-day rolling mean is a natural feature, but it is easy to write it wrong:

```python
df['y'].rolling(7).mean()          # WRONG: the window ends at day t, so it contains y[t]
df['y'].shift(1).rolling(7).mean() # RIGHT: the window ends at day t-1
```

In [ ]:
peek = pd.DataFrame({'y': y_all.iloc[:8]})
peek['roll3_WRONG'] = peek['y'].rolling(3).mean()             # includes y[t]
peek['roll3_RIGHT'] = peek['y'].shift(1).rolling(3).mean()    # ends at y[t-1]
peek.round(2)

Now the consequence: fit a model with lags 1–6 plus each version of the rolling mean.

In [ ]:
# The consequence, quantified. We give the model lags 1-6 plus a 7-day rolling mean.
leak = series.copy()
for L in range(1, 7):
    leak[f'lag_{L}'] = leak['y'].shift(L)
leak['roll7_wrong'] = leak['y'].rolling(7).mean()          # window ends at t   -> leaks
leak['roll7_right'] = leak['y'].shift(1).rolling(7).mean() # window ends at t-1 -> fine
leak = leak.dropna()

l_tr = leak.index <= TRAIN_END
l_va = (leak.index > TRAIN_END) & (leak.index <= VAL_END)
lag_cols = [f'lag_{L}' for L in range(1, 7)]

print('--- One feature, two definitions ---')
for col in ['roll7_wrong', 'roll7_right']:
    cols = lag_cols + [col]
    m = LinearRegression().fit(leak.loc[l_tr, cols], leak.loc[l_tr, 'y'])
    s = evaluate(leak.loc[l_va, 'y'], m.predict(leak.loc[l_va, cols]), col)
    print(f'  {col:<12} MAE={s["MAE"]:8.4f}   MASE={s["MASE"]:7.4f}   R2={s["R2"]:8.5f}')

print('\n  The leaking model is not good. It is cheating: since')
print('    roll7_wrong = (y[t] + y[t-1] + ... + y[t-6]) / 7')
print('  and lags 1-6 are also features, least squares can solve for y[t] exactly:')
print('    y[t] = 7 * roll7_wrong - (y[t-1] + ... + y[t-6])')
print('  It has not learned to forecast. It has learned to do arithmetic.')

> ### Rule 3: `shift(1)` before every rolling window. For every feature ask: *was this known on that date?*

### The feature set

| Group | Features | Captures |
|-------|----------|----------|
| Trend | `t` | Long-run growth |
| Month | 11 month dummies (January is the reference) | Annual cycle |
| Weekday | 6 day-of-week dummies (Monday is the reference) | Weekly pattern |
| Lags | $y_{t-1}, y_{t-2}, y_{t-3}, y_{t-7}, y_{t-14}$ | Recent values |
| Rolling | mean of the previous 7 and 28 days | Local level |

We drop one dummy per group to avoid the dummy trap (notes 03). Calendar features need no shift: the month and weekday of a future date are known in advance.

In [ ]:
LAGS = [1, 2, 3, 7, 14]
ROLL_WINDOWS = [7, 28]


def build_features(df, lags=LAGS, roll_windows=ROLL_WINDOWS):
    """Build a causal design matrix from a frame with columns ['y', 't'].

    Every column is computable from information strictly before the row's own
    date, except the calendar columns, which are known arbitrarily far ahead.
    """
    X = pd.DataFrame(index=df.index)

    # --- calendar: known for any future date ---
    X['trend'] = df['t']
    for month in range(2, 13):                  # month 1 (January) is the reference
        X[f'month_{month}'] = (df.index.month == month).astype(float)
    for d in range(1, 7):                       # d = 0 (Monday) is the reference
        X[f'dow_{d}'] = (df.index.dayofweek == d).astype(float)

    # --- lags: strictly past values ---
    for L in lags:
        X[f'lag_{L}'] = df['y'].shift(L)

    # --- rolling: shift(1) FIRST, then the window ---
    for w in roll_windows:
        X[f'rmean_{w}'] = df['y'].shift(1).rolling(w).mean()

    return X


X_all = build_features(series)
design = X_all.join(series['y']).dropna()       # drop warm-up rows with NaN lags

# Re-derive the masks on the trimmed index so every model below uses the same rows.
d_train = design.index <= TRAIN_END
d_val   = (design.index > TRAIN_END) & (design.index <= VAL_END)
d_test  = design.index > VAL_END

FEATURES = list(X_all.columns)
CALENDAR = [c for c in FEATURES if c.startswith(('trend', 'month_', 'dow_'))]

X_tr, y_tr = design.loc[d_train, FEATURES], design.loc[d_train, 'y']
X_va, y_va = design.loc[d_val,   FEATURES], design.loc[d_val,   'y']
X_te, y_te = design.loc[d_test,  FEATURES], design.loc[d_test,  'y']

print(f'--- Design matrix ---')
print(f'  Features : {len(FEATURES)}   (of which {len(CALENDAR)} calendar)')
print(f'  Rows     : {len(design)} (lost {len(series) - len(design)} to the warm-up window)')
print(f'  train {X_tr.shape}   val {X_va.shape}   test {X_te.shape}')
design[['trend', 'month_2', 'dow_1', 'lag_1', 'rmean_7', 'y']].head(3).round(2)

---
## Fitting

### Model 1: calendar only

Trend + month + weekday, nothing from the series itself. Since we generated the data, we can check that the coefficients match the truth. This is a great sanity check for the feature code.

In [ ]:
results = []

m1 = LinearRegression().fit(X_tr[CALENDAR], y_tr)
p1_va = m1.predict(X_va[CALENDAR])
results.append(evaluate(y_va, p1_va, '1. Calendar only'))

print('--- Recovered coefficients vs. the true generating process ---')
print(f'  trend  : {m1.coef_[CALENDAR.index("trend")]:.4f}   (true 0.0300)')
for d, day in zip(range(1, 7), ['Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']):
    true_effect = dow_effect[d] - dow_effect[0]
    est = m1.coef_[CALENDAR.index(f'dow_{d}')]
    print(f'  {day} vs Mon : {est:6.2f}   (true {true_effect:5.2f})')

Close to the truth. (The weekday estimates are all shifted by about 1: they are differences *relative to Monday*, so an error in Monday's level moves all of them together.)

### Model 2: adding lags and rolling means

In [ ]:
m2 = LinearRegression().fit(X_tr, y_tr)
results.append(evaluate(y_va, m2.predict(X_va), '2. + lags and rolling means'))

pd.DataFrame(results).set_index('model').round(3)

### Model 3: Ridge

Lags and rolling means are strongly correlated with each other, so we add L2 regularisation (notes 03). Ridge needs scaled features, and the scaler must be **fitted on the training block only**. Putting it inside a `Pipeline` does that for us.

In [ ]:
m3 = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X_tr, y_tr)
p3_va = m3.predict(X_va)
results.append(evaluate(y_va, p3_va, '3. Ridge (alpha=1, scaled)'))

pd.DataFrame(results).set_index('model').round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
window = slice(0, 120)                          # first four months of validation
ax.plot(y_va.index[window], y_va.values[window], 'o-', ms=3, lw=1,
        color='black', label='actual')
ax.plot(y_va.index[window], p1_va[window], lw=1.6, alpha=0.85, label='1. calendar only')
ax.plot(y_va.index[window], p3_va[window], lw=1.6, alpha=0.85, label='3. ridge + lags')
ax.set_title('One-step-ahead forecasts on the validation block', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## Mistake 3: Shuffled Cross-Validation

To tune Ridge's $\alpha$ we need cross-validation, but `KFold(shuffle=True)` has the same problem as the random split. The time-series version, `TimeSeriesSplit`, always trains on the past and validates on what comes next:

```
fold 1  [=====train=====][val]
fold 2  [========train========][val]
fold 3  [===========train===========][val]
```

In [ ]:
tscv = TimeSeriesSplit(n_splits=5, test_size=90)

fig, ax = plt.subplots(figsize=(13, 3.2))
for i, (tr, va) in enumerate(tscv.split(X_tr)):
    ax.plot(tr, np.full_like(tr, i, dtype=float), '|', color='steelblue', ms=8)
    ax.plot(va, np.full_like(va, i, dtype=float), '|', color='darkorange', ms=8)
ax.set_yticks(range(5))
ax.set_yticklabels([f'fold {i+1}' for i in range(5)])
ax.set_xlabel('index within the training block', fontsize=12)
ax.set_title('TimeSeriesSplit — expanding window, rolling origin '
             '(blue = train, orange = validate)', fontsize=14)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

We choose $\alpha$ with both splitters (inside the training block) and compare.

In [ ]:
ALPHAS = np.logspace(-3, 4, 25)


def cv_mae(alphas, splitter, X, y):
    """Mean CV MAE for each alpha under a given splitter."""
    scores = []
    for a in alphas:
        fold_scores = []
        for tr, va in splitter.split(X):
            pipe = make_pipeline(StandardScaler(), Ridge(alpha=a))
            pipe.fit(X.iloc[tr], y.iloc[tr])
            fold_scores.append(mean_absolute_error(y.iloc[va], pipe.predict(X.iloc[va])))
        scores.append(np.mean(fold_scores))
    return np.array(scores)


mae_ts = cv_mae(ALPHAS, TimeSeriesSplit(n_splits=5, test_size=90), X_tr, y_tr)
mae_kf = cv_mae(ALPHAS, KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE), X_tr, y_tr)

best_ts = ALPHAS[np.argmin(mae_ts)]
best_kf = ALPHAS[np.argmin(mae_kf)]

print('--- Selected penalty ---')
print(f'  TimeSeriesSplit : alpha = {best_ts:9.4f}   CV MAE = {mae_ts.min():.3f}')
print(f'  Shuffled KFold  : alpha = {best_kf:9.4f}   CV MAE = {mae_kf.min():.3f}  <- optimistic')

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.semilogx(ALPHAS, mae_ts, 'o-', ms=4, label='TimeSeriesSplit (valid)')
ax.semilogx(ALPHAS, mae_kf, 's--', ms=4, label='KFold shuffled (invalid)')
ax.axvline(best_ts, color='steelblue', ls=':', lw=1.5)
ax.set_xlabel(r'ridge penalty $\alpha$', fontsize=12)
ax.set_ylabel('CV MAE', fontsize=12)
ax.set_title('Hyperparameter selection: honest vs. leaking cross-validation', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

The shuffled curve is below the honest one for every $\alpha$ (too optimistic), and it can even pick a **different** $\alpha$. Leakage doesn't only report the wrong score, it can make you choose the wrong model.

> ### Rule 4: cross-validate with `TimeSeriesSplit`, never `KFold(shuffle=True)`.

In [ ]:
m4 = make_pipeline(StandardScaler(), Ridge(alpha=best_ts)).fit(X_tr, y_tr)
p4_va = m4.predict(X_va)
results.append(evaluate(y_va, p4_va, f'4. Ridge (alpha={best_ts:.3g}, TS-CV)'))

pd.DataFrame(results).set_index('model').round(3)

---
## Final Evaluation on the Test Year

The model is chosen. We refit it on **train + validation** and score it **once** on the untouched test block.

In [ ]:
X_full = design.loc[d_train | d_val, FEATURES]
y_full = design.loc[d_train | d_val, 'y']

final_model = make_pipeline(StandardScaler(), Ridge(alpha=best_ts)).fit(X_full, y_full)
p_test = final_model.predict(X_te)

seasonal_naive_test = series['y'].shift(7).loc[X_te.index]

final = pd.DataFrame([
    evaluate(y_te, seasonal_naive_test, 'Seasonal naive (baseline)'),
    evaluate(y_te, p_test, 'Final ridge model (h=1)'),
]).set_index('model')
display(final.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].plot(y_te.index, y_te.values, lw=1, color='black', label='actual')
axes[0].plot(y_te.index, p_test, lw=1, color='crimson', alpha=0.8, label='forecast')
axes[0].set_title('Test block — one-step-ahead forecasts', fontsize=13)
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(fontsize=11)

axes[1].scatter(y_te, p_test, s=14, alpha=0.6)
lims = [y_te.min(), y_te.max()]
axes[1].plot(lims, lims, 'r--', lw=1.5, label='perfect')
axes[1].set_xlabel('actual', fontsize=11)
axes[1].set_ylabel('predicted', fontsize=11)
axes[1].set_title('Predicted vs. actual', fontsize=13)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

Report it as: *"MASE of X at horizon 1, on a held-out year, against a seasonal naive baseline."*

Two warnings before using this model:

- **Horizon.** We always predicted **tomorrow** ($h=1$). To forecast 30 days ahead, `lag_1` does not exist yet, and the error grows with the horizon. **Rule 5: validate at the horizon you will actually use.**
- **Trend.** The `trend` feature assumes growth continues forever at the same rate. Validation can't warn you about this, as the plot below shows.

In [ ]:
far_dates = pd.date_range('2019-01-01', periods=int(9 * 365), freq='D')
far = pd.DataFrame({'y': np.nan, 't': np.arange(len(far_dates))}, index=far_dates)
far_pred = m1.predict(build_features(far)[CALENDAR])

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(series.index, series['y'], lw=0.8, label='observed', color='black', alpha=0.6)
ax.plot(far_dates, far_pred, lw=1.2, color='crimson', label='calendar-model extrapolation')
ax.axvline(series.index[-1], color='black', ls='--', lw=1)
ax.set_title('Linear trends extrapolate confidently and without limit', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Predicted demand on {far_dates[-1].date()}: {far_pred[-1]:.1f}'
      f'   (last observed: {series["y"].iloc[-1]:.1f})')

---
## Exercise: Which features matter?

Refit model 4 several times, each time removing one group (`trend`, month dummies, weekday dummies, lags, rolling means), and compare the validation MASE. Which removal hurts most?

In [ ]:
# Your turn:

<details><summary><b>Solution</b></summary>Run the cell below.</details>

In [ ]:
groups = {
    'trend':   ['trend'],
    'month':   [c for c in FEATURES if c.startswith('month_')],
    'dow':     [c for c in FEATURES if c.startswith('dow_')],
    'lags':    [c for c in FEATURES if c.startswith('lag_')],
    'rolling': [c for c in FEATURES if c.startswith('rmean_')],
}

rows = [evaluate(y_va, m4.predict(X_va), 'all features')]
for name, cols in groups.items():
    keep = [c for c in FEATURES if c not in cols]
    mdl = make_pipeline(StandardScaler(), Ridge(alpha=best_ts)).fit(X_tr[keep], y_tr)
    rows.append(evaluate(y_va, mdl.predict(X_va[keep]), f'without {name}'))

ablation = pd.DataFrame(rows).set_index('model')[['MAE', 'MASE']]
ablation['MASE delta'] = (ablation['MASE'] - ablation.loc['all features', 'MASE']).round(4)
ablation.round(3)

---
## Summary: the five rules

1. **Never shuffle.** Split in time order.
2. **Compare against a real baseline** (seasonal naive) with MASE, not $R^2$.
3. **Build features causally:** `shift(1)` before every rolling window.
4. **Cross-validate with a rolling origin** (`TimeSeriesSplit`).
5. **Validate at the horizon you will use.**